# Test: Is the Abnormality Detector Working?

This notebook is just for checking `services/abnormality_detector.py` by
eye. It does two things:

1. **Known-answer test** -- feeds it made-up test results where we already
   know what the answer *should* be, so you can compare "Expected" vs
   "Got" side by side.
2. **Real report test** -- feeds it real NER output already saved from one
   of your test report images, so you can see it run on messy real data.

This notebook does NOT call OCR or the NER model -- it only tests the
abnormality detector itself.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from services.abnormality_detector import AbnormalityDetector

detector = AbnormalityDetector()
print("Detector loaded.")

Detector loaded.


## Test 1: Known-answer check

Six made-up tests below, each designed to hit a different case:

| Test | What it checks |
|---|---|
| Hemoglobin | value just under the low end -> LOW / MILD |
| Glucose | value far above a "< 110" one-sided range -> HIGH / SEVERE |
| Cholesterol | non-numeric value ("abnormal") -> could_not_parse |
| WBC | value inside range -> NORMAL |
| Platelets | missing unit + range -> incomplete_data |
| Sodium | value inside range -> NORMAL |

If the "Got" column matches the "Expected" column for every row, the
detector is working correctly.

In [2]:
known_entities = {
    "TEST_NAME":  ["Hemoglobin",   "Glucose", "Cholesterol", "WBC",     "Platelets", "Sodium"],
    "TEST_VALUE": ["12.5",         "250",     "abnormal",    "7.0",     "150",       "140"],
    "TEST_UNIT":  ["g/dL",         "mg/dL",   "mg/dL",       "x10^9/L", None,        "mmol/L"],
    "REF_RANGE":  ["13.0 - 17.0",  "< 110",   "< 200",       "4.0-11.0", None,       "135 - 145"],
}

expected_status = ["LOW", "HIGH", "could_not_parse", "NORMAL", "incomplete_data", "NORMAL"]
expected_severity = ["MILD", "SEVERE", None, None, None, None]

In [3]:
known_results = detector.detect(known_entities)

df = pd.DataFrame(known_results)
df["expected_status"] = expected_status
df["expected_severity"] = expected_severity
df["status_ok"] = df["status"].fillna("NONE") == df["expected_status"].fillna("NONE")
df["severity_ok"] = df["severity"].fillna("NONE") == df["expected_severity"].fillna("NONE")

display(df[["test_name", "value", "status", "expected_status", "status_ok",
            "severity", "expected_severity", "severity_ok", "deviation_percent"]])

all_ok = df["status_ok"].all() and df["severity_ok"].all()
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED -- see status_ok / severity_ok columns above")

print()
print(detector.get_summary(known_results))

,test_name,value,status,expected_status,status_ok,severity,expected_severity,severity_ok,deviation_percent
0,Hemoglobin,12.5,LOW,LOW,True,MILD,MILD,True,3.846154
1,Glucose,250.0,HIGH,HIGH,True,SEVERE,SEVERE,True,127.272727
2,Cholesterol,NaN,could_not_parse,could_not_parse,True,None,None,True,NaN
3,WBC,7.0,NORMAL,NORMAL,True,None,None,True,NaN
4,Platelets,NaN,incomplete_data,incomplete_data,True,None,None,True,NaN
5,Sodium,140.0,NORMAL,NORMAL,True,None,None,True,NaN


ALL CHECKS PASSED

{'total_tests': 6, 'normal_count': 2, 'abnormal_count': 2, 'severe_count': 1, 'abnormal_tests': ['Hemoglobin', 'Glucose']}


## Test 2: Real report data

This loads NER output that was already extracted and saved from your real
test report images (`evaluation/test_results/ner_real_images_results.json`).
No OCR or NER model needs to run here -- we're reusing that saved output.

Note: a couple of these reports trip up the NER model itself (e.g. it
occasionally tags a stray word as a fake "test name"), which can shift the
test name/value/range pairing out of order. That's a NER issue, not an
abnormality detector issue -- this section is here so you can see what real
(imperfect) input looks like, not to prove the detector is broken.

In [4]:
with open(PROJECT_ROOT / "evaluation" / "test_results" / "ner_real_images_results.json", encoding="utf-8") as f:
    ner_results = json.load(f)

report_images = [name for name, r in ner_results.items() if r.get("true_label") == "Report"]
print(f"Found {len(report_images)} report images: {report_images}")

Found 9 report images: ['003img.jpg', '004img.jpeg', '010img.jpg', '011img.png', '015img.jpeg', '016img.jpg', '018img.jpg', '022img.jpeg', '023img.jpg']


In [5]:
IMAGE_NAME = report_images[0]  # change this to try a different report image

entry = ner_results[IMAGE_NAME]
grouped = {}
for e in entry["entities"]:
    grouped.setdefault(e["entity_type"], []).append(e["text"])

print(f"Testing on: {IMAGE_NAME}")
for key in ("TEST_NAME", "TEST_VALUE", "TEST_UNIT", "REF_RANGE"):
    print(f"{key}: {grouped.get(key, [])}")

Testing on: 003img.jpg
TEST_NAME: ['urea', 'uv', 'creatinine', 'complete blood count', 'whole blood hemoglobin ( hb )', 'red blood cells', 'hematocrit', 'pcv', 'mean corpuscular volume', 'mcv', 'mean corpuscular hemoglobin', 'mch', 'mean', 'corpuscular hemoglobin', 'red cell distribution width', 'rdw', 'white blood cells', 'wbc', 'neutrophils']
TEST_VALUE: ['16 . 00', '0 . 50', '10 . 5', '4 . 09', '32', '79 . 7', '25 . 6', '32 . 1', '12 . 4', '13 . 51', '86', '8']
TEST_UNIT: ['l', 'mg / dl', 'mg / dl', 'g / dl', 'ml / ul', '%', 'fi', 'pg', 'g / dl', '%', 'thousand / μl', '%']
REF_RANGE: ['16 . 600 - 48 . 500', '0 . 700 - 1 . 300', '11 - 13 . 5', '3 . 5 - 5', '0', '33 - 39', '75 - 87', '24 - 30', '31 - 35', '12 - 15', '5 - 15 . 5', '25 - 45']


In [6]:
real_results = detector.detect(grouped)
display(pd.DataFrame(real_results))
print()
print(detector.get_summary(real_results))

,test_name,value,unit,ref_range,status,severity,deviation_percent
0,urea,16.00,l,16 . 600 - 48 . 500,LOW,MILD,3.614458
1,uv,0.50,mg / dl,0 . 700 - 1 . 300,LOW,MODERATE,28.571429
2,creatinine,10.50,mg / dl,11 - 13 . 5,LOW,MILD,4.545455
3,complete blood count,4.09,g / dl,3 . 5 - 5,NORMAL,None,NaN
4,whole blood hemoglobin ( hb ),32.00,ml / ul,0,could_not_parse,None,NaN
5,red blood cells,79.70,%,33 - 39,HIGH,SEVERE,104.358974
6,hematocrit,25.60,fi,75 - 87,LOW,SEVERE,65.866667
7,pcv,32.10,pg,24 - 30,HIGH,MILD,7.000000
8,mean corpuscular volume,12.40,g / dl,31 - 35,LOW,SEVERE,60.000000
9,mcv,13.51,%,12 - 15,NORMAL,None,NaN



{'total_tests': 19, 'normal_count': 2, 'abnormal_count': 9, 'severe_count': 5, 'abnormal_tests': ['urea', 'uv', 'creatinine', 'red blood cells', 'hematocrit', 'pcv', 'mean corpuscular volume', 'mean corpuscular hemoglobin', 'mch']}


In [7]:
# Quick summary across every report image, so you can spot-check all of them at once
for name in report_images:
    entry = ner_results[name]
    grouped = {}
    for e in entry["entities"]:
        grouped.setdefault(e["entity_type"], []).append(e["text"])
    results = detector.detect(grouped)
    summary = detector.get_summary(results)
    print(f"{name:15s} total={summary['total_tests']:3d}  normal={summary['normal_count']:3d}  "
          f"abnormal={summary['abnormal_count']:3d}  severe={summary['severe_count']:3d}")

003img.jpg      total= 19  normal=  2  abnormal=  9  severe=  5
004img.jpeg     total= 10  normal=  0  abnormal=  0  severe=  0
010img.jpg      total=  7  normal=  0  abnormal=  2  severe=  2
011img.png      total=  0  normal=  0  abnormal=  0  severe=  0
015img.jpeg     total=  6  normal=  1  abnormal=  2  severe=  2
016img.jpg      total=  7  normal=  0  abnormal=  0  severe=  0
018img.jpg      total= 17  normal=  2  abnormal=  1  severe=  1
022img.jpeg     total=  7  normal=  1  abnormal=  3  severe=  1
023img.jpg      total= 15  normal=  1  abnormal=  2  severe=  1
